In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.0 MB/s eta 0:00:00


In [4]:
import yaml

yaml_path = '/content/drive/MyDrive/VisDrone_Dataset/visdrone.yaml'

# Load the existing content
with open(yaml_path, 'r') as f:
    yaml_content = yaml.safe_load(f)

# Update the 'path' to the absolute path on Google Drive
yaml_content['path'] = '/content/drive/MyDrive/VisDrone_Dataset'

# Save the corrected YAML back to Drive
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("Updated YAML configuration:")
import pprint
pprint.pprint(yaml_content)

Updated YAML configuration:
{'names': {0: 'human',
           1: 'bicycle',
           2: 'car',
           3: 'van',
           4: 'truck',
           5: 'tricycle',
           6: 'awning-tricycle',
           7: 'bus',
           8: 'motor'},
 'nc': 9,
 'path': '/content/drive/MyDrive/VisDrone_Dataset',
 'test': 'VisDrone2019-DET-test-dev/images',
 'train': 'VisDrone2019-DET-train/images',
 'val': 'VisDrone2019-DET-val/images'}


### Speeding up Training: Copying Dataset to Local Runtime
To avoid the slow Google Drive file streaming, we will copy the dataset to the local `/content/` directory.

In [5]:
import shutil
import os

# Source on Drive and Destination on local disk
source_dir = '/content/drive/MyDrive/VisDrone_Dataset'
dest_dir = '/content/VisDrone_Dataset'

if not os.path.exists(dest_dir):
    print(f"Copying dataset from Drive to {dest_dir}... This may take a few minutes.")
    shutil.copytree(source_dir, dest_dir)
    print("Copy complete!")
else:
    print("Dataset already exists locally.")

Copying dataset from Drive to /content/VisDrone_Dataset... This may take a few minutes.
Copy complete!


In [ ]:
import shutil
import os
import yaml

# 1. Define paths
source_dataset = '/content/drive/MyDrive/VisDrone_Dataset'
dest_dataset = '/content/VisDrone_Dataset'
local_yaml = '/content/visdrone_migration.yaml'

# 2. Copy Dataset if not already present
if not os.path.exists(dest_dataset):
    print(f"Copying dataset from {source_dataset} to {dest_dataset}... This may take a while.")
    shutil.copytree(source_dataset, dest_dataset)
    print("Dataset copy complete.")
else:
    print("Dataset already exists locally.")

# 3. Modify and Save Local YAML
# We load the config to update the base 'path' to the local directory
drive_yaml = os.path.join(source_dataset, 'visdrone.yaml')
if os.path.exists(drive_yaml):
    with open(drive_yaml, 'r') as f:
        config = yaml.safe_load(f)

    # Update the root path to the local directory we just created
    config['path'] = dest_dataset

    with open(local_yaml, 'w') as f:
        yaml.dump(config, f)
    print(f"Local YAML created and modified at: {local_yaml}")
else:
    print(f"Error: {drive_yaml} not found.")

In [ ]:
from ultralytics import YOLO
import torch

# Confirm device
device = 0 if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Load the YOLO11 medium model
model = YOLO('yolo11m.pt')

# Start Memory-Safe Optimized Training
# Configuration: 30 Epochs, imgsz=896, batch=4
results = model.train(
    data='/content/visdrone_local.yaml',
    epochs=30,
    imgsz=640,
    batch=4,
    device=device,
    workers=2,
    cache=True,
    amp=True,
    # Augmentations enabled for best accuracy
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    flipud=0.5,
    fliplr=0.5,
    degrees=10.0,
    # Management
    project='/content/drive/MyDrive/runs',
    name='yolo11m_visdrone_final',
    save_period=10,
    patience=25,
    exist_ok=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: 0
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/visdrone_local.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hs